# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nauman024/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

Archetype-to-Action Mapping & Reason Codes:

REWRITE_METAS_AND_REFRESH (Priority 1 — High Severity)

- Reason Code: HIGH_CTR_DEFICIT_AND_STALE

- Condition: High impression volume ($>300$), average ranking position $\le 15$, but CTR $<1.5\%$ with active age $>180$ days.

- Action: Rewrite title tags, update meta descriptions, and refresh hero content section.

UPDATE_CONTENT_BODY (Priority 2 — Moderate Severity)

- Reason Code: MODERATE_POSITION_DRIFT
- Condition: Position rank dropped from top 5 to lower page 1/page 2 ($10 < \text{position} \le 20$), impression volume stable.
- Action: Audit subheadings (H2/H3), add updated statistics, and improve internal linking.

MONITOR (Priority 3 — Low Risk)
- Reason Code: STABLE_PERFORMANCE
- Condition: Page maintains healthy tier-level CTR and stable position rank.
- Action: No immediate manual intervention; keep in automated tracking queue.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

Intended Use:

- Designed as a decision-support priorization tool for SEO leads and editorial content teams.

- Generates a ranked queue of candidate pages that exhibit measured underperformance flags relative to peer baseline expectations.

Known Limitations:

- No Causal Guarantee: The underlying scoring model measures statistical associations in historical search performance data; it does not model search engine algorithm internals directly.

- Unobserved External Factors: Does not account for sudden off-page events (e.g., domain backlink loss, brand PR spikes, or seasonal demand shifts).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

Human Review Rules:

- Every candidate page flagged for REWRITE_METAS_AND_REFRESH or UPDATE_CONTENT_BODY must undergo manual editorial review prior to live publication.

- Verify search intent alignment on Google SERPs before executing content changes.

The NO-GO List (What NOT to Automate):

1. Automated AI Content Rewriting: Never auto-publish machine-generated content directly to live production URLs without human editorial verification.

2. Automated URL Redirects / Deletions: Never automate 301 redirects or page deletions based purely on risk scores.

3. Brand-Critical Pages: Core product landing pages and high-conversion commercial pages are strictly excluded from automated bulk changes.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

Monitoring Strategy & Retrain Triggers:

- Drift Monitoring: Track monthly distributions of average position and CTR across client panels.
- Retrain Trigger 1 (Data Drift): Retrain model when core metric feature distributions deviate by $>15\%$ relative to the baseline training snapshot.
- Retrain Trigger 2 (Performance Decay): Trigger model retraining if validation F1-score drops below $0.65$ on rolling monthly data splits.
- Retrain Trigger 3 (Cadence): Scheduled quarterly refresh of training datasets.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import os
import duckdb
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from google.colab import userdata

# 1. Connect to DuckDB & Hugging Face
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# 2. Extract Data from Mid-Panel Month (2026-03)
query = f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    AVG(f.gsc_avg_position) as avg_position,
    SUM(f.gsc_impressions) as total_impressions,
    SUM(f.gsc_clicks) as total_clicks,
    (SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0)) as ctr,
    DATEDIFF('day', MIN(f.report_date), MAX(f.report_date)) + 30 as active_days
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('{rel}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
WHERE c.is_deleted IS FALSE
GROUP BY f.client_hash_id, f.content_hash_id
HAVING SUM(f.gsc_impressions) > 100;
"""

df = con.sql(query).df().fillna(0)

# 3. Compute Action Scores with Dynamic Percentile Cutoffs
df['ctr_expected'] = np.where(df['avg_position'] <= 10, 0.05, 0.01)
df['ctr_deficit'] = np.maximum(0, df['ctr_expected'] - df['ctr'])
df['action_score'] = np.clip((df['ctr_deficit'] * 2000) + (df['active_days'] / 5), 0, 100).round(2)

# Calculate 80th and 40th percentiles to guarantee all 3 classes populate
p80 = df['action_score'].quantile(0.80)
p40 = df['action_score'].quantile(0.40)

def assign_playbook_action(row):
    if row['action_score'] >= p80:
        return 'REWRITE_METAS_AND_REFRESH', 'HIGH_CTR_DEFICIT_AND_STALE'
    elif row['action_score'] >= p40:
        return 'UPDATE_CONTENT_BODY', 'MODERATE_POSITION_DRIFT'
    else:
        return 'MONITOR', 'STABLE_PERFORMANCE'

df[['action_label', 'reason_code']] = df.apply(assign_playbook_action, axis=1, result_type='expand')
df_playbook = df.sort_values(by='action_score', ascending=False)

# 4. Export Artifacts
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Export Queue CSV
csv_path = 'work/outputs/action_playbook_queue.csv'
df_playbook[['content_hash_id', 'client_hash_id', 'avg_position', 'ctr', 'action_score', 'reason_code', 'action_label']].to_csv(csv_path, index=False)
print(f"Playbook Queue exported to {csv_path} ({len(df_playbook):,} rows).")

# Export Metrics JSON (Receipts)
metrics = {
    "total_pages_evaluated": int(len(df_playbook)),
    "rewrite_refresh_count": int((df_playbook['action_label'] == 'REWRITE_METAS_AND_REFRESH').sum()),
    "update_body_count": int((df_playbook['action_label'] == 'UPDATE_CONTENT_BODY').sum()),
    "monitor_count": int((df_playbook['action_label'] == 'MONITOR').sum()),
    "mean_action_score": float(df_playbook['action_score'].mean())
}

metrics_path = 'work/outputs/w07_playbook_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Playbook metrics exported to {metrics_path}.")

# 5. Export Figure for Paper
plt.figure(figsize=(8, 5))
df_playbook['action_label'].value_counts().plot(kind='bar', color=['#d9534f', '#f0ad4e', '#5cb85c'])
plt.title('Action Queue Distribution (Week 7 Playbook)')
plt.xlabel('Action Label')
plt.ylabel('Number of Pages')
plt.xticks(rotation=15)
plt.tight_layout()

fig_path = 'work/figures/w07_action_queue_distribution.png'
plt.savefig(fig_path, dpi=300)
plt.close()
print(f"Figure exported to {fig_path}.")

print("\n=== Playbook Action Distribution Preview ===")
display(df_playbook['action_label'].value_counts().to_frame())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Playbook Queue exported to work/outputs/action_playbook_queue.csv (101,203 rows).
Playbook metrics exported to work/outputs/w07_playbook_metrics.json.
Figure exported to work/figures/w07_action_queue_distribution.png.

=== Playbook Action Distribution Preview ===


,count
action_label,
REWRITE_METAS_AND_REFRESH,46539
UPDATE_CONTENT_BODY,31023
MONITOR,23641


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.